In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install SpeechRecognition
!pip install pydub
!pip install accelerate
!pip install bitsandbytes
!pip install pyngrok
!pip install flask_cors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
import os
import math
import time
import speech_recognition as sr
from pydub import AudioSegment
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import random
from flask import Flask, request, jsonify
from pyngrok import ngrok
from flask_cors import CORS, cross_origin
import cv2
import numpy as np
import tensorflow as tf
import logging
from werkzeug.utils import secure_filename
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import AutoTokenizer, LlamaForCausalLM

In [4]:
def download_mp3(url, save_path):
    """Download an MP3 file from a URL."""
    response = requests.get(url)

    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"Download successful. File saved to {save_path}")
        return True
    else:
        print(f"Failed to download. Status code: {response.status_code}")
        return False

In [5]:
def preprocess_audio(input_audio_file, output_audio_file=None):
    """Preprocess audio to improve transcription quality and speed."""
    if output_audio_file is None:
        filename, ext = os.path.splitext(input_audio_file)
        output_audio_file = f"{filename}_processed{ext}"

    # Load the audio
    audio = AudioSegment.from_file(input_audio_file)

    # Normalize volume (increase volume for better recognition)
    audio = audio.normalize(headroom=1.0)

    # Convert to mono (if stereo)
    if audio.channels > 1:
        audio = audio.set_channels(1)

    # Set sample rate to 16kHz (good for speech recognition)
    audio = audio.set_frame_rate(16000)

    # Export processed audio
    audio.export(output_audio_file, format=os.path.splitext(output_audio_file)[1][1:])

    print(f"Audio preprocessed and saved to {output_audio_file}")
    return output_audio_file


In [6]:
def split_audio_with_overlap(input_audio_file, chunk_duration=30, overlap_duration=3):
    """Split audio into overlapping chunks to prevent losing words at boundaries."""
    print(f"Splitting audio file with {overlap_duration}s overlap: {input_audio_file}")
    audio = AudioSegment.from_file(input_audio_file)

    # Calculate total duration in seconds
    total_duration_sec = len(audio) / 1000

    # Calculate effective chunk duration (accounting for overlap)
    effective_chunk_duration = chunk_duration - overlap_duration

    # Calculate total number of chunks
    total_chunks = math.ceil(total_duration_sec / effective_chunk_duration)

    print(f"Audio duration: {total_duration_sec:.2f}s, Total chunks: {total_chunks}")

    # Create a directory to store the chunks if it doesn't exist
    if not os.path.exists('audio_chunks'):
        os.makedirs('audio_chunks')

    # Split the audio file into chunks with overlap and save them
    chunks_info = []
    for i in range(total_chunks):
        # Calculate start and end times with overlap
        start_time = max(0, i * effective_chunk_duration * 1000)
        end_time = min((i * effective_chunk_duration + chunk_duration) * 1000, len(audio))

        chunk = audio[start_time:end_time]

        # Normalize each chunk for better recognition
        chunk = chunk.normalize(headroom=1.0)

        output_path = f'audio_chunks/chunk_{i}.wav'
        chunk.export(output_path, format='wav')

        # Store info about this chunk for later processing
        chunks_info.append({
            'index': i,
            'file_path': output_path,
            'start_time': start_time / 1000,  # Convert to seconds
            'end_time': end_time / 1000,      # Convert to seconds
            'duration': (end_time - start_time) / 1000  # Convert to seconds
        })

    # Save chunk info for debugging
    with open('chunks_info.json', 'w') as f:
        json.dump(chunks_info, f, indent=2)

    print(f"Audio split into {total_chunks} overlapping chunks")
    return chunks_info


In [7]:
def transcribe_single_chunk(chunk_info, retries=3, delay=2):
    """Transcribe a single audio chunk with retries using only Google service."""
    chunk_index = chunk_info['index']
    audio_file_path = chunk_info['file_path']
    recognizer = sr.Recognizer()

    # Configure recognizer for better performance
    recognizer.energy_threshold = 300  # Lower threshold for detecting speech
    recognizer.dynamic_energy_threshold = True
    recognizer.pause_threshold = 0.8  # Shorter pause threshold

    # Try multiple attempts with increasing backoff
    for attempt in range(retries):
        try:
            with sr.AudioFile(audio_file_path) as audio_file:
                audio_data = recognizer.record(audio_file)

                # Only use Google's service
                try:
                    chunk_transcript = recognizer.recognize_google(audio_data, language="en-US")
                    print(f"✓ Chunk {chunk_index+1}: Successfully transcribed (attempt {attempt+1})")
                    return (chunk_index, chunk_transcript, chunk_info)
                except sr.UnknownValueError:
                    if attempt == retries - 1:
                        print(f"⚠ Chunk {chunk_index+1}: Unable to recognize speech")
                        return (chunk_index, "", chunk_info)
                except sr.RequestError as e:
                    print(f"⚠ Chunk {chunk_index+1}: Google API error: {e}")
                    if "quota" in str(e).lower():
                        # If quota exceeded, use longer delay
                        delay = 5
        except Exception as e:
            print(f"⚠ Chunk {chunk_index+1}: Error: {str(e)}")

        # Wait before retrying with randomized exponential backoff
        if attempt < retries - 1:
            # Calculate backoff with jitter to avoid thundering herd
            backoff_time = delay * (2 ** attempt) * (0.5 + random.random()/2)
            print(f"Retrying chunk {chunk_index+1} in {backoff_time:.1f}s (attempt {attempt+2}/{retries})...")
            time.sleep(backoff_time)

    # If all attempts failed
    return (chunk_index, "", chunk_info)

def parallel_transcribe_chunks(chunks_info, max_workers=None, batch_size=5):
    """Transcribe multiple audio chunks in batches to avoid rate limiting."""
    start_time = time.time()

    # If max_workers not specified, use a reasonable default
    if max_workers is None:
        import multiprocessing
        max_workers = min(multiprocessing.cpu_count(), 3)  # Limit workers to reduce API rate issues

    print(f"Starting transcription with {max_workers} workers in batches of {batch_size}")

    # Store all results
    all_chunk_results = []

    # Process chunks in batches to avoid overwhelming the API
    for batch_start in range(0, len(chunks_info), batch_size):
        batch_end = min(batch_start + batch_size, len(chunks_info))
        batch = chunks_info[batch_start:batch_end]

        print(f"\nProcessing batch {batch_start//batch_size + 1}/{math.ceil(len(chunks_info)/batch_size)} " +
              f"(chunks {batch_start+1}-{batch_end})...")

        chunk_results = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit batch of transcription tasks
            futures = {
                executor.submit(transcribe_single_chunk, chunk_info): chunk_info['index']
                for chunk_info in batch
            }

            # Process results as they complete
            for future in as_completed(futures):
                chunk_index, chunk_text, chunk_info = future.result()
                chunk_results.append({
                    'index': chunk_index,
                    'text': chunk_text,
                    'start_time': chunk_info['start_time'],
                    'end_time': chunk_info['end_time']
                })
                print(f"Progress: {len(chunk_results)}/{len(batch)} chunks in current batch completed")

        # Add batch results to all results
        all_chunk_results.extend(chunk_results)

        # Save intermediate results after each batch
        with open(f'transcription_batch_{batch_start//batch_size + 1}.json', 'w') as f:
            json.dump(chunk_results, f, indent=2)

        # Small delay between batches to avoid API rate limiting
        if batch_end < len(chunks_info):
            delay = 3
            print(f"Waiting {delay}s before next batch to avoid rate limiting...")
            time.sleep(delay)

    # Sort all results by index to maintain order
    all_chunk_results.sort(key=lambda x: x['index'])

    # Save raw transcription results for debugging
    with open('raw_transcription_results.json', 'w') as f:
        json.dump(all_chunk_results, f, indent=2)

    # Smart merging of overlapping chunks
    complete_transcript = smart_merge_transcripts(all_chunk_results)

    elapsed_time = time.time() - start_time
    print(f"Transcription completed in {elapsed_time:.2f} seconds")

    return complete_transcript


In [8]:
def smart_merge_transcripts(chunk_results):
    """Merge transcripts with handling for overlaps and empty chunks."""
    # Filter out empty chunks
    non_empty_chunks = [chunk for chunk in chunk_results if chunk['text'].strip()]

    if not non_empty_chunks:
        return ""

    # For simple case with no overlap, just join the texts
    if len(non_empty_chunks) == 1:
        return non_empty_chunks[0]['text']

    # Initialize with first chunk
    merged_text = non_empty_chunks[0]['text']

    # Merge remaining chunks with overlap detection
    for i in range(1, len(non_empty_chunks)):
        current_chunk = non_empty_chunks[i]['text']

        if not current_chunk:
            continue

        previous_chunk = merged_text

        # Look for overlap - try different word lengths to find matching boundaries
        found_overlap = False

        # Try matching last N words of previous with first N words of current
        for overlap_words in range(5, 0, -1):
            prev_words = previous_chunk.split()[-overlap_words:] if len(previous_chunk.split()) >= overlap_words else []
            curr_words = current_chunk.split()[:overlap_words] if len(current_chunk.split()) >= overlap_words else []

            if prev_words and curr_words:
                prev_overlap = ' '.join(prev_words).lower()
                curr_overlap = ' '.join(curr_words).lower()

                # If we find overlap, merge accordingly
                if prev_overlap == curr_overlap:
                    # Add current chunk without the overlapping part
                    merged_text = previous_chunk + ' ' + ' '.join(current_chunk.split()[overlap_words:])
                    found_overlap = True
                    break

        # If no overlap found, just append with space
        if not found_overlap:
            merged_text = previous_chunk + ' ' + current_chunk

    return merged_text

In [9]:
def load_models():
    # Load PEGASUS model for summarization
    model_path = '/content/drive/MyDrive/PEGASUS_FINETUNED'
    pegasus_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    pegasus_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Load DeepSeek model for Q&A
    deepseek_model_name = "deepseek-ai/deepseek-llm-7b-base"
    deepseek_tokenizer = AutoTokenizer.from_pretrained(deepseek_model_name)
    deepseek_model = LlamaForCausalLM.from_pretrained(
        pretrained_model_name_or_path=deepseek_model_name,
        load_in_8bit=True,
        device_map={'': 0},
    )

    return pegasus_model, pegasus_tokenizer, deepseek_model, deepseek_tokenizer


In [10]:
app = Flask(__name__)
app.config["UPLOAD_FOLDER"] = "downloads"
os.makedirs(app.config["UPLOAD_FOLDER"], exist_ok=True)

# Enable CORS
CORS(app)

# Set up logging
logging.basicConfig(level=logging.INFO)

# Global variables
summary_cache = ""  # To hold the PEGASUS summary
transcript_cache = ""  # To hold the full transcript

# Load models at startup
pegasus_model, pegasus_tokenizer, deepseek_model, deepseek_tokenizer = load_models()

# Set ngrok authentication token
ngrok.set_auth_token("2cOOhCeMaiDYjZWi1b1NGCYfLnO_5FqABvBX85mm9gNZf5jnK")

# Connect to ngrok and get public URL
port_no = 5000  # Set the port number
public_url = ngrok.connect(port_no).public_url
print(f"To access the Global link please click {public_url}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


...

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.97G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

To access the Global link please click https://8217-34-125-59-169.ngrok-free.app


In [11]:
@app.route("/transcribe", methods=["POST"])
@cross_origin(origin='*')
def transcribe_audio():
    global summary_cache, transcript_cache

    data = request.get_json()
    if not data or "url" not in data:
        return jsonify({"error": "Missing 'url' in request body"}), 400

    url = data["url"]
    save_path = os.path.join(app.config["UPLOAD_FOLDER"], "audio_input.mp3")

    # Step 1: Download
    if not download_mp3(url, save_path):
        return jsonify({"error": "Failed to download audio file"}), 500
    logging.info("Audio downloaded successfully.")

    try:
        # Step 2–4: Process audio & transcribe
        logging.info("Processing audio...")
        processed_audio = preprocess_audio(save_path)

        logging.info("Splitting into chunks...")
        chunks_info = split_audio_with_overlap(processed_audio, chunk_duration=30, overlap_duration=3)

        logging.info("Transcribing chunks...")
        transcript = parallel_transcribe_chunks(chunks_info, max_workers=3, batch_size=5)
        transcript_cache = transcript  # Save for later use

        # Save transcript
        transcript_path = os.path.join(app.config["UPLOAD_FOLDER"], "transcript_output.txt")
        with open(transcript_path, "w") as f:
            f.write(transcript)

        # Step 5: Summarization using PEGASUS
        logging.info("Generating summary with PEGASUS...")
        inputs = pegasus_tokenizer(transcript, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = pegasus_model.generate(
            inputs.input_ids,
            num_beams=4,
            min_length=75,
            max_length=150,
            length_penalty=0.5,
            early_stopping=True
        )
        summary = pegasus_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        summary_cache = summary  # Store for use in other endpoints

        return jsonify({
            "message": "Transcription and summarization successful",
            "summary": summary
        }), 200

    except Exception as e:
        logging.exception("Error in processing pipeline.")
        return jsonify({"error": str(e)}), 500

In [12]:
@app.route("/qa", methods=["POST"])
@cross_origin(origin='*')
def qa_from_summary():
    global summary_cache, transcript_cache

    data = request.get_json()
    if not data or "question" not in data:
        return jsonify({"error": "Missing 'question' in request body"}), 400

    if not summary_cache:
        return jsonify({"error": "No summary available. Please run /transcribe first."}), 400

    try:
        # Use DeepSeek model for dynamic Q&A responses
        # Create a prompt that includes both summary and full transcript for context
        prompt = f"""You are a podcast assistant that helps users understand podcast content.

Context:
Basic summary: {summary_cache}

Full transcript: {transcript_cache[:3000]}... [truncated for brevity]

User question: {data['question']}

Provide a specific, detailed answer to the user's question based on the podcast content above.
"""
        logging.info("Generating dynamic response with DeepSeek model...")

        # Tokenize and generate response
        input_ids = deepseek_tokenizer(prompt, return_tensors="pt").input_ids.cuda()
        output = deepseek_model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.95
        )

        # Decode the response and remove the prompt
        full_response = deepseek_tokenizer.decode(output[0], skip_special_tokens=True)

        # Extract only the model's answer, removing the prompt
        answer_start = full_response.find("User question:") + len(f"User question: {data['question']}")
        response_text = full_response[answer_start:].strip()

        return jsonify({"answer": response_text}), 200

    except Exception as e:
        logging.exception("QA generation failed.")
        return jsonify({"error": str(e)}), 500


In [ ]:
if __name__ == "__main__":
    port_no = 5000
    app.run(port=port_no)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:28:01] "OPTIONS /transcribe HTTP/1.1" 200 -


Download successful. File saved to downloads/audio_input.mp3
Audio preprocessed and saved to downloads/audio_input_processed.mp3
Splitting audio file with 3s overlap: downloads/audio_input_processed.mp3
Audio duration: 1201.63s, Total chunks: 45
Audio split into 45 overlapping chunks
Starting transcription with 3 workers in batches of 5

Processing batch 1/9 (chunks 1-5)...
✓ Chunk 2: Successfully transcribed (attempt 1)
Progress: 1/5 chunks in current batch completed
✓ Chunk 1: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch completed
✓ Chunk 3: Successfully transcribed (attempt 1)
Progress: 3/5 chunks in current batch completed
✓ Chunk 4: Successfully transcribed (attempt 1)
Progress: 4/5 chunks in current batch completed
✓ Chunk 5: Successfully transcribed (attempt 1)
Progress: 5/5 chunks in current batch completed
Waiting 3s before next batch to avoid rate limiting...

Processing batch 2/9 (chunks 6-10)...
✓ Chunk 6: Successfully transcribed (attempt 1)
P

Download successful. File saved to downloads/audio_input.mp3
Audio preprocessed and saved to downloads/audio_input_processed.mp3
Splitting audio file with 3s overlap: downloads/audio_input_processed.mp3
Audio duration: 1201.63s, Total chunks: 45
Audio split into 45 overlapping chunks
Starting transcription with 3 workers in batches of 5

Processing batch 1/9 (chunks 1-5)...
✓ Chunk 2: Successfully transcribed (attempt 1)
Progress: 1/5 chunks in current batch completed
✓ Chunk 1: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch completed
✓ Chunk 3: Successfully transcribed (attempt 1)
Progress: 3/5 chunks in current batch completed
✓ Chunk 4: Successfully transcribed (attempt 1)
Progress: 4/5 chunks in current batch completed
✓ Chunk 5: Successfully transcribed (attempt 1)
Progress: 5/5 chunks in current batch completed
Waiting 3s before next batch to avoid rate limiting...

Processing batch 2/9 (chunks 6-10)...
✓ Chunk 6: Successfully transcribed (attempt 1)
P

INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:34:03] "POST /transcribe HTTP/1.1" 200 -


...

INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:35:53] "OPTIONS /transcribe HTTP/1.1" 200 -


✓ Chunk 28: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch completed
✓ Chunk 26: Successfully transcribed (attempt 1)
Progress: 3/5 chunks in current batch completed
Download successful. File saved to downloads/audio_input.mp3
Audio preprocessed and saved to downloads/audio_input_processed.mp3
Splitting audio file with 3s overlap: downloads/audio_input_processed.mp3
Audio duration: 145.20s, Total chunks: 6
Audio split into 6 overlapping chunks
Starting transcription with 3 workers in batches of 5

Processing batch 1/2 (chunks 1-5)...
✓ Chunk 29: Successfully transcribed (attempt 1)
Progress: 4/5 chunks in current batch completed
✓ Chunk 30: Successfully transcribed (attempt 1)
Progress: 5/5 chunks in current batch completed
Waiting 3s before next batch to avoid rate limiting...
✓ Chunk 1: Successfully transcribed (attempt 1)
Progress: 1/5 chunks in current batch completed
✓ Chunk 2: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch c

INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:37:10] "POST /transcribe HTTP/1.1" 200 -


✓ Chunk 37: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch completed
✓ Chunk 36: Successfully transcribed (attempt 1)
Progress: 3/5 chunks in current batch completed
✓ Chunk 40: Successfully transcribed (attempt 1)
Progress: 4/5 chunks in current batch completed
✓ Chunk 39: Successfully transcribed (attempt 1)
Progress: 5/5 chunks in current batch completed
Waiting 3s before next batch to avoid rate limiting...

Processing batch 9/9 (chunks 41-45)...
Retrying chunk 43 in 1.4s (attempt 2/3)...
✓ Chunk 41: Successfully transcribed (attempt 1)
Progress: 1/5 chunks in current batch completed
✓ Chunk 42: Successfully transcribed (attempt 1)
Progress: 2/5 chunks in current batch completed
✓ Chunk 45: Successfully transcribed (attempt 1)
Progress: 3/5 chunks in current batch completed
Retrying chunk 43 in 3.3s (attempt 3/3)...
Retrying chunk 44 in 1.4s (attempt 2/3)...
⚠ Chunk 43: Unable to recognize speech
Progress: 4/5 chunks in current batch completed
Retrying c

INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:38:44] "POST /transcribe HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:39:45] "OPTIONS /qa HTTP/1.1" 200 -
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
INFO:werkzeug:127.0.0.1 - - [15/Apr/2025 21:40:09] "POST /qa HTTP/1.1" 200 -
